In [2]:
import json
import pandas as pd
import time
import re
import ast
import requests
import shutil
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support import expected_conditions as EC

In [3]:
CHROME_BINARY = shutil.which("chromium")
CHROMEDRIVER_PATH = shutil.which("chromedriver")

chrome_options = Options()
chrome_options.binary_location = CHROME_BINARY

chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument("--disable-infobars")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--window-size=1200,800")
chrome_options.add_argument("--blink-settings=imagesEnabled=false")

prefs = {
    "profile.managed_default_content_settings.images": 2,
    "profile.managed_default_content_settings.stylesheets": 2,
    "profile.managed_default_content_settings.fonts": 2,
    "profile.managed_default_content_settings.plugins": 2,
    "profile.managed_default_content_settings.notifications": 2,
}
chrome_options.add_experimental_option("prefs", prefs)

# Selenium 4 way to set capabilities:
chrome_options.set_capability("pageLoadStrategy", "eager")

service = Service(CHROMEDRIVER_PATH)
driver = webdriver.Chrome(service=service, options=chrome_options)

In [5]:
url = "https://www.bilbasen.dk/brugt/bil/xpeng/g9/performance-5d/6768517"

In [6]:
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "da-DK,da;q=0.9,en;q=0.8",
}

r = requests.get(url, headers=headers, timeout=30)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

items = [el.get_text(strip=True) for el in soup.find_all(attrs={"data-e2e": "car-equipment-item"})]
items = list(dict.fromkeys(items))  # de-dupe, keep order

print(f"Found {len(items)} items")
print(items[:30])

Found 67 items
['2 zone klima', 'Højdejust. forsæde', '21 tommer Alufælge', 'Isofix', '360° kamera', 'Klimaanlæg, 2-zonet', '4x el-ruder', 'Kunstlæder', 'ABS-bremser', 'Kunstlæderindtræk', 'Adaptiv fartpilot', 'Læderrat', 'Airbags', 'Luftundervogn', 'Alufælge', 'Massage i forsæder', 'Ambiente belysning', 'Mørktonede ruder i bag', 'Antispin', 'Multifunktionsrat', 'Aut. nedbl. bakspejl', 'Musikstreaming via bluetooth', 'Auto hold', 'Navigation', 'Auto. nødbremse', 'Nøglefri adgang', 'Automatgear', 'Nøglefri tænding', 'Automatisk lys', 'Panoramatag']


In [1]:
import re
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE = "https://www.bilbasen.dk"
BRAND = "xpeng"
FUEL = "el"  # adjust if your fuel param differs
TARGET = "https://www.bilbasen.dk/brugt/bil/xpeng/g9/performance-5d/6768517"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "da-DK,da;q=0.9,en;q=0.8",
}

def get_soup(url: str, session: requests.Session, retries: int = 3) -> BeautifulSoup:
    last = None
    for i in range(retries):
        try:
            r = session.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return BeautifulSoup(r.text, "html.parser")
        except Exception as e:
            last = e
            time.sleep(0.8 * (i + 1))
    raise last

def parse_pages(soup: BeautifulSoup) -> int:
    tag = soup.find("span", {"data-e2e": "pagination-total"})
    if not tag:
        return 1
    m = re.search(r"\d+", tag.get_text(strip=True))
    return int(m.group(0)) if m else 1

def extract_listing_links(soup: BeautifulSoup) -> list[str]:
    links = []
    listing_article_re = re.compile(r"^Listing_listing")
    listing_link_re = re.compile(r"^Listing_link")

    for art in soup.find_all("article", class_=listing_article_re):
        for a in art.find_all("a", class_=listing_link_re, href=True):
            links.append(urljoin(BASE, a["href"]))
    return links

def download_brand_requests(brand: str, fuel: str) -> list[str]:
    base_url = f"{BASE}/brugt/bil/{brand}?fuel={fuel}&includeengroscvr=true&includeleasing=false"

    with requests.Session() as session:
        soup1 = get_soup(base_url, session)
        max_page = parse_pages(soup1)

        all_links = []
        for page in range(1, max_page + 1):
            url = f"{base_url}&page={page}"
            print(f"Fetching {brand} page {page}/{max_page}: {url}")
            soup = get_soup(url, session)
            all_links.extend(extract_listing_links(soup))

    # de-dupe keep order
    return list(dict.fromkeys(all_links))

if __name__ == "__main__":
    links = download_brand_requests(BRAND, FUEL)
    print("Total links:", len(links))
    print("First 10:", links[:10])
    print("Target found:", TARGET in links)


Fetching xpeng page 1/3: https://www.bilbasen.dk/brugt/bil/xpeng?fuel=el&includeengroscvr=true&includeleasing=false&page=1
Fetching xpeng page 2/3: https://www.bilbasen.dk/brugt/bil/xpeng?fuel=el&includeengroscvr=true&includeleasing=false&page=2
Fetching xpeng page 3/3: https://www.bilbasen.dk/brugt/bil/xpeng?fuel=el&includeengroscvr=true&includeleasing=false&page=3
Total links: 55
First 10: ['https://www.bilbasen.dk/brugt/bil/xpeng/p7/long-range-4d/6673048', 'https://www.bilbasen.dk/brugt/bil/xpeng/g9/standard-range-5d/6742598', 'https://www.bilbasen.dk/brugt/bil/xpeng/g6/performance-5d/6763502', 'https://www.bilbasen.dk/brugt/bil/xpeng/p7/long-range-4d/6704967', 'https://www.bilbasen.dk/brugt/bil/xpeng/g6/performance-5d/6719687', 'https://www.bilbasen.dk/brugt/bil/xpeng/g6/long-range-5d/6709600', 'https://www.bilbasen.dk/brugt/bil/xpeng/g9/performance-5d/6748590', 'https://www.bilbasen.dk/brugt/bil/xpeng/g6/long-range-5d/6762568', 'https://www.bilbasen.dk/brugt/bil/xpeng/g6/standard-

In [9]:
import json
import re
import requests
from bs4 import BeautifulSoup

URL = "https://www.bilbasen.dk/brugt/bil/xpeng/g9/performance-5d/6768517"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "da-DK,da;q=0.9,en;q=0.8",
}

PROPS_RE = re.compile(r"var\s*_props\s*=\s*({.*?})\s*;", flags=re.DOTALL)

def extract_props_dict(url: str) -> dict:
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()

    # fast path: regex across full HTML
    m = PROPS_RE.search(r.text)
    if m:
        return json.loads(m.group(1))

    # fallback: look through script tags
    soup = BeautifulSoup(r.text, "html.parser")
    for s in soup.find_all("script"):
        txt = (s.get_text() or "").lstrip()
        if txt.startswith("var _props"):
            m2 = PROPS_RE.search(txt)
            if m2:
                return json.loads(m2.group(1))

    raise RuntimeError("No var _props JSON found on page")

if __name__ == "__main__":
    props = extract_props_dict(URL)
    print("OK - parsed _props JSON")
    print("Top-level keys:", list(props.keys())[:20])


OK - parsed _props JSON
Top-level keys: ['listing', 'writeToSellerOptions', 'tracking', 'relatedListings', 'financing', 'headMeta', 'breadCrumbs', 'popularSearches', 'environment', 'turnstileKey', 'experiments']


In [10]:
props

{'listing': {'externalId': 6768517,
  'syiId': '9d1f00f7-33c7-40cd-9614-9304f96f146a',
  'tenant': 'bilinfo',
  'canonicalUrl': 'https://www.bilbasen.dk/brugt/bil/xpeng/g9/performance-5d/6768517',
  'price': {'name': 'Kontantpris', 'displayValue': '499.900 kr.'},
  'vehicle': {'make': 'Xpeng',
   'model': 'G9',
   'variant': 'Performance 5d',
   'modelYear': 2024,
   'ratings': {'average': 4.85,
    'numberOfReviews': 10,
    'subRatings': [{'name': 'Køreegenskaber', 'rating': 4.8},
     {'name': 'Driftsomkostninger', 'rating': 4.7},
     {'name': 'Sikkerhed', 'rating': 5.0},
     {'name': 'Værdi for pengene', 'rating': 4.9}],
    'label': 'populær'},
   'details': [{'name': 'Modelår', 'displayValue': '2024'},
    {'name': '1. registrering',
     'displayValue': '5/2024',
     'tooltip': {'title': '1. registrering',
      'text': '1. registrering er den måned og det år, hvor bilen første gang har fået nummerplade på. Produceret er den måned og det år, bilen er blevet produceret. Modelå

In [11]:
import pandas as pd

d = props  # the object you pasted

# Flatten dict structure (but lists stay as lists)
base = pd.json_normalize(d, sep=".")

# Turn listing.vehicle.details -> columns like detail__Modelår, detail__Kilometertal, ...
details = d.get("listing", {}).get("vehicle", {}).get("details", [])
details_cols = {
    f"detail__{x.get('name')}": x.get("displayValue")
    for x in details
}

# Turn listing.vehicle.ratings.subRatings -> columns like subrating__Sikkerhed, ...
subratings = d.get("listing", {}).get("vehicle", {}).get("ratings", {}).get("subRatings", [])
subrating_cols = {
    f"subrating__{x.get('name')}": x.get("rating")
    for x in subratings
}

# Add those columns to the base row
wide = base.assign(**details_cols, **subrating_cols)

# (Optional) drop the original list columns if you don’t want them
wide = wide.drop(columns=[
    "listing.vehicle.details",
    "listing.vehicle.ratings.subRatings",
], errors="ignore")

wide


,breadCrumbs,popularSearches,turnstileKey,experiments,listing.externalId,listing.syiId,listing.tenant,listing.canonicalUrl,listing.price.name,listing.price.displayValue,...,detail__Periodisk afgift,detail__Ydelse,detail__Acceleration,detail__Tophastighed,detail__Trækvægt,detail__Farve,subrating__Køreegenskaber,subrating__Driftsomkostninger,subrating__Sikkerhed,subrating__Værdi for pengene
0,"[{'url': '/brugt/bil/xpeng', 'text': 'Xpeng', ...",[],0x4AAAAAAACCg4xUUb0XNJBP,"[{'experimentName': 'BilbasenAbtest', 'variant...",6768517,9d1f00f7-33c7-40cd-9614-9304f96f146a,bilinfo,https://www.bilbasen.dk/brugt/bil/xpeng/g9/per...,Kontantpris,499.900 kr.,...,840 kr. / år,551 hk/717 nm,"3,9 sek.",200 km/t,1.500 kg,Sortmetal,4.8,4.7,5.0,4.9


In [12]:
el_cols = [
        'scrape_timestamp','price.displayValue', 'Nypris', 'vehicle.make', 'vehicle.model', 'vehicle.variant', 'vehicle.modelYear', '1. registrering', 
        'Kilometertal', 'Ydelse', 'Acceleration', 'Tophastighed', 'Trækvægt', 'Farve',
        'Kategori', 'Type', 'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne', 'Max. trækvægt m/bremse', 'Trækhjul',
        'Drivmiddel', 'Energiforbrug', 'Batterikapacitet', 'Rækkevidde', 'Hjemmeopladning AC', 'Hurtig opladning DC', 'Opladningstid DC 10-80%',
        'Airbags', 'ABS-bremser', 'ESP', 'Døre', 'Periodisk afgift', 
        'price.description', 'seller.name', 'seller.address.zipCode', 'seller.address.city', 'seller.sellerOtherItems.numberOfListings',
        'vehicle.ratings.average', 'vehicle.ratings.numberOfReviews', 'canonicalUrl','externalId', 'description'
    ]

In [13]:
wide[el_cols]

KeyError: "None of [Index(['scrape_timestamp', 'price.displayValue', 'Nypris', 'vehicle.make',\n       'vehicle.model', 'vehicle.variant', 'vehicle.modelYear',\n       '1. registrering', 'Kilometertal', 'Ydelse', 'Acceleration',\n       'Tophastighed', 'Trækvægt', 'Farve', 'Kategori', 'Type',\n       'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne',\n       'Max. trækvægt m/bremse', 'Trækhjul', 'Drivmiddel', 'Energiforbrug',\n       'Batterikapacitet', 'Rækkevidde', 'Hjemmeopladning AC',\n       'Hurtig opladning DC', 'Opladningstid DC 10-80%', 'Airbags',\n       'ABS-bremser', 'ESP', 'Døre', 'Periodisk afgift', 'price.description',\n       'seller.name', 'seller.address.zipCode', 'seller.address.city',\n       'seller.sellerOtherItems.numberOfListings', 'vehicle.ratings.average',\n       'vehicle.ratings.numberOfReviews', 'canonicalUrl', 'externalId',\n       'description'],\n      dtype='object')] are in the [columns]"